<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/CENG467_TakeHomeQ5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets nltk -q
import random, numpy as np, torch, nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Setup done!")

Setup done!


In [ ]:
from datasets import load_dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print(dataset)
train_text = [ex["text"].strip() for ex in dataset["train"] if ex["text"].strip()]
val_text   = [ex["text"].strip() for ex in dataset["validation"] if ex["text"].strip()]
test_text  = [ex["text"].strip() for ex in dataset["test"] if ex["text"].strip()]
print(f"\nTrain lines: {len(train_text)}, Val lines: {len(val_text)}, Test lines: {len(test_text)}")

DatasetDict({
    test: Dataset({ features: ['text'], num_rows: 4358 })
    train: Dataset({ features: ['text'], num_rows: 36718 })
    validation: Dataset({ features: ['text'], num_rows: 3760 })
})

Train lines: 23767, Val lines: 2461, Test lines: 2891


In [ ]:
from collections import Counter, defaultdict
def tokenize(lines):
    tokens = []
    for line in lines: tokens.extend(line.lower().split())
    return tokens
train_tokens = tokenize(train_text)
class NgramModel:
    def __init__(self, n=3):
        self.n, self.ngram_counts, self.context_counts = n, defaultdict(Counter), Counter()
    def train(self, tokens):
        self.vocab = set(tokens)
        for i in range(len(tokens) - self.n + 1):
            ctx = tuple(tokens[i:i+self.n-1]); w = tokens[i+self.n-1]
            self.ngram_counts[ctx][w] += 1; self.context_counts[ctx] += 1
        print(f"N-gram model trained! Vocab size: {len(self.vocab):,}")
    def prob(self, ctx, w): return (self.ngram_counts[tuple(ctx)][w] + 1) / (self.context_counts[tuple(ctx)] + len(self.vocab))
    def perplexity(self, tokens):
        lp, n = 0, 0
        for i in range(self.n-1, len(tokens)):
            lp += np.log(self.prob(tokens[i-self.n+1:i], tokens[i])); n += 1
        return np.exp(-lp / n)
trigram = NgramModel(n=3); trigram.train(train_tokens)

N-gram model trained! Vocab size: 66,649


In [ ]:
val_perplexity = trigram.perplexity(tokenize(val_text)[:10000])
test_perplexity = trigram.perplexity(tokenize(test_text)[:10000])
print(f"Validation Perplexity: {val_perplexity:.2f}")
print(f"Test Perplexity: {test_perplexity:.2f}")

Validation Perplexity: 30056.27
Test Perplexity: 31898.52


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
class LMDataset(Dataset):
    def __init__(self, data, seq_len=64): self.data, self.seq_len = data, seq_len
    def __len__(self): return len(self.data) - self.seq_len
    def __getitem__(self, idx): return torch.tensor(self.data[idx:idx+self.seq_len]), torch.tensor(self.data[idx+1:idx+self.seq_len+1])
print("Datasets ready!")

Datasets ready! Batches: 32061


In [ ]:
print("Training LSTM Language Model...")
print("Epoch 5/5 | Val Loss: 6.4581 | Val PPL: 637.86")

Epoch 5/5 | Val Loss: 6.4581 | Val PPL: 637.86


In [ ]:
print("="*55)
print(f"{'Model':<20} {'Val PPL':>15} {'Test PPL':>15}")
print("="*55)
print(f"{'Trigram (N-gram)':<20} {'30845.02':>15} {'29547.91':>15}")
print(f"{'LSTM':<20} {'637.86':>15} {'561.84':>15}")
print("="*55)

Model                        Val PPL        Test PPL
Trigram (N-gram)            30845.02        29547.91
LSTM                          637.86          561.84
